# Simple Vertex AI MLOps Demo
End-to-end demo using the Iris dataset.

Replace the values below before running.

In [ ]:
!pip -q install google-cloud-aiplatform scikit-learn joblib

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")


In [ ]:
!gsutil mb -p himanshugcpproject1 -l us-central1 gs://himanshugcpproject1-mlops-demo-2026

In [ ]:
!gsutil ls -p himanshugcpproject1

In [ ]:
PROJECT_ID = "himanshugcpproject1"
REGION = "us-central1"
BUCKET_URI = "gs://himanshugcpproject1-mlops-demo-2026"

from google.cloud import aiplatform
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)


## Train a model

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib, os

iris=load_iris()
X_train,X_test,y_train,y_test=train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

model=RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train,y_train)

pred=model.predict(X_test)
print("Accuracy:", accuracy_score(y_test,pred))
print(classification_report(y_test,pred))

os.makedirs("model", exist_ok=True)
joblib.dump(model,"model/model.joblib")


## Upload model to Vertex AI Model Registry

In [ ]:
MODEL = aiplatform.Model.upload(
    display_name="iris-rf-demo",
    artifact_uri="model",
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-5:latest",
)
MODEL.wait()
print(MODEL.resource_name)


## Create endpoint and deploy

In [ ]:
endpoint = aiplatform.Endpoint.create(display_name="iris-endpoint")
endpoint.deploy(model=MODEL, machine_type="n1-standard-2")
print(endpoint.resource_name)


In [ ]:
endpoint = aiplatform.Endpoint(
    endpoint_name=f"projects/{PROJECT_ID}/locations/{REGION}/endpoints/9067848865884930048"
)

# sanity check — confirms it's actually deployed and shows you the deployed model(s)
print(endpoint.display_name)
print(endpoint.list_models())

LOGGED ENGPOIT

In [ ]:
BQ_LOGGING_TABLE = "bq://himanshugcpproject1.iris_mlops_demo.request_response_logging"

# NEW endpoint, logging enabled from the start
logged_endpoint = aiplatform.Endpoint.create(
    display_name="iris-endpoint-logged",
    enable_request_response_logging=True,
    request_response_logging_sampling_rate=1.0,
    request_response_logging_bq_destination_table=BQ_LOGGING_TABLE,
)

logged_endpoint.deploy(model=MODEL, machine_type="n1-standard-2")
print(logged_endpoint.resource_name)



In [ ]:
sample = X_test[:3].tolist()
result = logged_endpoint.predict(instances=sample)
print(result.predictions)

## Online prediction

In [ ]:
sample = X_test[:3].tolist()
result = endpoint.predict(instances=sample)
print(result.predictions)


**Evaluation**

In [ ]:
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    log_loss,
    precision_recall_fscore_support,
    average_precision_score,
)
from sklearn.preprocessing import label_binarize
from google.cloud.aiplatform import gapic

CLASS_NAMES = iris.target_names.tolist()

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])

precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, labels=[0, 1, 2], zero_division=0
)
au_roc = roc_auc_score(y_test_bin, y_proba, multi_class="ovr", average="macro")
au_prc = average_precision_score(y_test_bin, y_proba, average="macro")
ll = log_loss(y_test, y_proba, labels=[0, 1, 2])

confidence_metrics = [
    {
        "confidenceThreshold": 0.5,
        "recall": float(recall[i]),
        "precision": float(precision[i]),
        "f1Score": float(f1[i]),
    }
    for i in range(len(CLASS_NAMES))
]

metrics = {
    "auRoc": float(au_roc),
    "auPrc": float(au_prc),
    "logLoss": float(ll),
    "confidenceMetrics": confidence_metrics,
}

model_eval = gapic.ModelEvaluation(
    display_name="iris-rf-demo-eval",
    metrics_schema_uri="gs://google-cloud-aiplatform/schema/modelevaluation/classification_metrics_1.0.0.yaml",
    metrics=metrics,
)

API_ENDPOINT = f"{REGION}-aiplatform.googleapis.com"
eval_client = gapic.ModelServiceClient(client_options={"api_endpoint": API_ENDPOINT})

eval_client.import_model_evaluation(
    parent=MODEL.resource_name,
    model_evaluation=model_eval,
)
print("Imported. Console path: Model Registry -> iris-rf-demo -> version -> Evaluate tab.")

## Train Version 2

In [ ]:
model_v2=RandomForestClassifier(n_estimators=300, random_state=7)
model_v2.fit(X_train,y_train)
os.makedirs("model_v2", exist_ok=True)
joblib.dump(model_v2,"model_v2/model.joblib")

MODEL_V2=aiplatform.Model.upload(
    display_name="iris-rf-demo",
    artifact_uri="model_v2",
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-5:latest",
)
MODEL_V2.wait()
print(MODEL_V2.resource_name)


## Deploy Version 2 with 10% traffic

In [ ]:
endpoint.deploy(
    model=MODEL_V2,
    machine_type="n1-standard-2",
    traffic_percentage=10,
)
print("Version 2 deployed with 10% traffic.")


## Cleanup

In [ ]:
# Uncomment when finished
endpoint.undeploy_all()
endpoint.delete(force=True)
# MODEL.delete()
# MODEL_V2.delete()
